<a href="https://colab.research.google.com/github/brendonhuynhbp-hub/gt-markets/blob/main/notebooks/3.2/40_backtest_strategies.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ======================================================================
# Notebook: 40_backtest_strategies.ipynb
# Purpose : Backtest multiple strategy families on predictions from run 3.2
# ======================================================================

# === 0) Imports & Setup ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import json, os, math, warnings, datetime as dt
warnings.filterwarnings("ignore")

# Colab Drive mount (safe if running locally)
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

# Project paths
PROJECT_DIR = Path("/content/drive/MyDrive/gt-markets")
RUN_ID_SRC  = "3.2"   # read predictions from here
RUN_ID_BT   = "3.3"   # write backtests here (new folder)

SRC_RUN    = PROJECT_DIR / "outputs" / "runs" / RUN_ID_SRC
SRC_SUM    = SRC_RUN / "60_summary"
SRC_INPUTS = SRC_RUN / "00_meta" / "input"   # cached inputs from training run

BT_RUN     = PROJECT_DIR / "outputs" / "runs" / RUN_ID_BT
BT_META    = BT_RUN / "00_meta"
BT_SUM     = BT_RUN / "60_summary"
BT_PLOTS   = BT_RUN / "70_plots"

for p in [BT_META, BT_SUM, BT_PLOTS]:
    p.mkdir(parents=True, exist_ok=True)

print("Reading predictions from:", SRC_SUM)
print("Writing backtest outputs to:", BT_RUN)

# Persist basic backtest config
bt_config = {
    "source_run_id": RUN_ID_SRC,
    "backtest_run_id": RUN_ID_BT,
    "created_at": dt.datetime.now().isoformat(timespec="seconds"),
    "notes": "Backtests read 3.2 predictions and write all artefacts to 3.3."
}
with open(BT_META / "backtest_config.json", "w") as f:
    json.dump(bt_config, f, indent=2)

# === 1) Parameters & constants ===

ASSETS = ["BTC", "GOLD", "OIL", "USDCNY"]
FREQS  = ["D", "W"]            # Daily & Weekly
DATASETS_TO_USE = ["eng"]      # focus ENG first; can add "base","ext" later

# Cost assumptions (per side, as bps)
COST_BPS = {
    "BTC": 10,     # 0.10% round-trip ≈ 20bps
    "GOLD": 3,
    "OIL": 3,
    "USDCNY": 1,
}

# Annualisation factors
AF = {"D": 252, "W": 52}

# Big-move thresholds on absolute returns (as fractions, not %)
BIGMOVE_TAU = {
    ("BTC","D"):0.04, ("BTC","W"):0.06,
    ("GOLD","D"):0.010, ("GOLD","W"):0.015,
    ("OIL","D"):0.02, ("OIL","W"):0.025,
    ("USDCNY","D"):0.002, ("USDCNY","W"):0.004,
}

# Strategy parameter grids (first cut – sensible but not huge)
CLS_THRESHOLDS = [0.55, 0.60]
REG_K_VALUES   = [2.0, 3.0]    # position ~ pred_ret/k
HYBRID_WEIGHTS = [0.25, 0.5, 0.75]

# Vol targeting (off by default)
USE_VOL_TARGET = False
TARGET_VOL_ANNUAL = 0.10
VOL_LOOKBACK = {"D": 20, "W": 20}

# === 2) Load predictions & helper data ===
preds = pd.read_csv(SRC_SUM / "predictions_master.csv", parse_dates=["date"])
leader = pd.read_csv(SRC_SUM / "leaderboards_master.csv")
print("Predictions loaded:", preds.shape)

# Map asset → close column name in cached input
ASSET_CLOSE_COLS = {
    "BTC":    "BTC-USD Close",
    "GOLD":   "GC=F Close",
    "OIL":    "CL=F Close",
    "USDCNY": "USDCNY=X Close",
}

# Choose ENG cached input if present, else fallback to base
ENG_FILE = None
for cand in ["merged_financial_trends_engineered_2025-09-07.csv",
             "merged_financial_trends_data_2025-09-07.csv"]:
    p = SRC_INPUTS / cand
    if p.exists():
        ENG_FILE = p
        break
assert ENG_FILE is not None, "No cached merged CSV found in source run inputs."

def load_eng_df(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df.dropna(subset=["Date"]).set_index("Date").sort_index()
    df = df.loc[~df.index.duplicated(keep="last")]
    return df

df_full = load_eng_df(ENG_FILE)

def resample_close(df: pd.DataFrame, asset: str, freq: str) -> pd.DataFrame:
    """Return frame with Close and simple returns at the requested frequency."""
    close_col = ASSET_CLOSE_COLS[asset]
    assert close_col in df.columns, f"Missing close column for {asset}: {close_col}"
    px = df[[close_col]].copy()
    px_res = px.resample(freq).last().dropna()
    px_res.rename(columns={close_col: "Close"}, inplace=True)
    px_res["ret_pct"] = px_res["Close"].pct_change() * 100.0   # percentage points
    return px_res

# === 3) TA indicator helpers (Close-only)
def SMA(series: pd.Series, n: int) -> pd.Series:
    return series.rolling(n, min_periods=n).mean()

def RSI(series: pd.Series, n: int = 14) -> pd.Series:
    delta = series.diff()
    up = delta.clip(lower=0.0)
    down = -delta.clip(upper=0.0)
    roll_up = up.rolling(n, min_periods=n).mean()
    roll_down = down.rolling(n, min_periods=n).mean()
    rs = roll_up / (roll_down.replace(0, np.nan))
    rsi = 100.0 - (100.0 / (1.0 + rs))
    return rsi

def MACD(series: pd.Series, fast=12, slow=26, sig=9):
    ema_fast = series.ewm(span=fast, adjust=False).mean()
    ema_slow = series.ewm(span=slow, adjust=False).mean().dropna()
    macd = ema_fast - ema_slow
    signal = macd.ewm(span=sig, adjust=False).mean()
    hist = macd - signal
    return macd, signal, hist

# === 4) Strategy engines ===

def cls_threshold_position(prob_up: pd.Series, theta: float) -> pd.Series:
    """+1 if prob_up >= theta, -1 if <= (1-theta), else 0."""
    pos = pd.Series(0.0, index=prob_up.index)
    pos[prob_up >= theta] = 1.0
    pos[prob_up <= (1.0 - theta)] = -1.0
    return pos

def reg_sized_position(pred_ret_pp: pd.Series, k: float) -> pd.Series:
    """Scale position by predicted % / k, clipped to [-1,1]."""
    pos = (pred_ret_pp / k).clip(-1.0, 1.0)
    return pos

def bigmove_filter(pos: pd.Series, pred_ret_pp: pd.Series, tau_abs: float) -> pd.Series:
    """Keep positions only where |pred_ret| >= tau; tau is fraction (e.g., 0.02 for 2%)."""
    mask = pred_ret_pp.abs().div(100.0) >= tau_abs
    return pos.where(mask, 0.0)

def ta_rules_from_prices(px_df: pd.DataFrame):
    """
    Compute TA signals from Close.
    Returns dict of position series in [-1, 0, +1] or fraction for MAs.
    """
    c = px_df["Close"]
    rsi = RSI(c, 14)
    sma10 = SMA(c, 10)
    sma50 = SMA(c, 50)
    macd, macds, _ = MACD(c, 12, 26, 9)

    # RSI long/short
    pos_rsi = pd.Series(0.0, index=c.index)
    pos_rsi[rsi < 30] = 1.0
    pos_rsi[rsi > 70] = -1.0

    # SMA cross (10/50)
    pos_sma = pd.Series(0.0, index=c.index)
    pos_sma[(sma10 > sma50)] = 1.0
    pos_sma[(sma10 < sma50)] = -1.0

    # MACD signal cross
    pos_macd = pd.Series(0.0, index=c.index)
    pos_macd[(macd > macds)] = 1.0
    pos_macd[(macd < macds)] = -1.0

    return {
        "TA_RSI14": pos_rsi,
        "TA_SMA10_50": pos_sma,
        "TA_MACD": pos_macd,
    }

def hybrid_confirm(pos_ml: pd.Series, pos_ta: pd.Series) -> pd.Series:
    """Take ML position only if TA agrees on direction; else 0."""
    agree = np.sign(pos_ml) == np.sign(pos_ta)
    return pos_ml.where(agree, 0.0)

def hybrid_weight(pos_ml: pd.Series, pos_ta: pd.Series, alpha: float) -> pd.Series:
    """Blend ML and TA positions."""
    return (alpha * pos_ml + (1.0 - alpha) * pos_ta).clip(-1.0, 1.0)

# === 5) Execution model, costs, and PnL ===

def apply_vol_target(pos: pd.Series, realized_ret_pp: pd.Series, freq: str, target_ann: float, lookback: int) -> pd.Series:
    """
    Scale positions so that position * return has approx target volatility.
    Vol estimated from rolling std of returns (percentage points → convert to fraction).
    """
    rf = realized_ret_pp / 100.0
    roll_std = rf.rolling(lookback, min_periods=lookback).std()
    ann_std = roll_std * math.sqrt(AF[freq])
    scale = (target_ann / ann_std).clip(0.0, 5.0)   # cap scaling to avoid blow-ups
    scale = scale.reindex(pos.index).fillna(0.0)
    return (pos * scale).clip(-2.0, 2.0)

def cost_rate_for(asset: str) -> float:
    """Round-trip cost model: per-step cost = cost_bps * |Δpos| (per side)."""
    bps = COST_BPS.get(asset, 5)
    return bps / 10000.0

def simulate_pnl(dates: pd.DatetimeIndex,
                 pos: pd.Series,
                 y_true_ret_pp: pd.Series,
                 asset: str,
                 freq: str,
                 use_vol_target: bool = False) -> pd.DataFrame:
    """
    Core backtest calculator.
    - pos is the signal decided at time t.
    - realised return usable is y_true_ret_pp at t (next period return relative to signal time).
    - costs on position changes.
    """
    pos = pos.reindex(dates).fillna(0.0)
    rets = y_true_ret_pp.reindex(dates).fillna(0.0) / 100.0

    if use_vol_target:
        pos = apply_vol_target(pos, y_true_ret_pp.reindex(dates).fillna(0.0), freq, TARGET_VOL_ANNUAL, VOL_LOOKBACK[freq])

    # Trading costs on position change
    dpos = pos.diff().fillna(pos)
    costs = cost_rate_for(asset) * dpos.abs()

    # Strategy return before costs
    strat_ret = pos * rets
    pnl_after_cost = strat_ret - costs

    out = pd.DataFrame({
        "ret": rets.values,
        "pos": pos.values,
        "pnl": pnl_after_cost.values,
        "cost": costs.values,
    }, index=dates)
    out["cum_pnl"] = (1.0 + out["pnl"]).cumprod()
    return out

# === 6) Metrics ===

def max_drawdown(equity: pd.Series) -> float:
    peak = equity.cummax()
    dd = (equity / peak) - 1.0
    return float(dd.min())

def ann_return(pnl: pd.Series, freq: str) -> float:
    # Compounded growth rate per period annualised
    growth = (1.0 + pnl).prod()
    n = len(pnl)
    if n == 0:
        return 0.0
    cagr = growth ** (AF[freq] / n) - 1.0
    return float(cagr)

def sharpe(pnl: pd.Series, freq: str) -> float:
    if pnl.std(ddof=0) == 0 or len(pnl) < 2:
        return 0.0
    return float(pnl.mean() / pnl.std(ddof=0) * math.sqrt(AF[freq]))

def turnover(dpos: pd.Series) -> float:
    return float(dpos.abs().sum())

def trade_stats(pos: pd.Series, pnl: pd.Series) -> dict:
    # Count trades when sign of position changes or flips from/to zero
    sgn = np.sign(pos.fillna(0.0))
    trade_points = (sgn != sgn.shift(1)).astype(int)
    trades = int(trade_points.sum())
    wins = int((pnl > 0).sum())
    losses = int((pnl < 0).sum())
    win_rate = wins / max(1, (wins + losses))
    return {"trades": trades, "win_rate": win_rate}

def bigmove_stats(y_true_ret_pp: pd.Series, pnl: pd.Series, asset: str, freq: str) -> dict:
    tau = BIGMOVE_TAU.get((asset, freq), 0.02)
    big = (y_true_ret_pp.abs() / 100.0) >= tau
    if big.sum() == 0:
        return {"bigmove_precision": np.nan, "bigmove_recall": np.nan, "bigmove_pnl_share": np.nan}
    hits = (pnl > 0) & big
    precision = hits.sum() / max(1, (pnl > 0).sum())
    recall = hits.sum() / big.sum()
    pnl_share = pnl[big].sum() / max(1e-12, pnl.sum())
    return {
        "bigmove_precision": float(precision),
        "bigmove_recall": float(recall),
        "bigmove_pnl_share": float(pnl_share),
    }

# === 7) Backtest loop ===

results_rows = []
signals_out_frames = []   # optional trade-level dump per strategy for inspection

for dataset in DATASETS_TO_USE:
    for asset in ASSETS:
        for freq in FREQS:

            # Slice predictions for this panel
            sub = preds[
                (preds["dataset"] == dataset) &
                (preds["asset"] == asset) &
                (preds["freq"] == freq)
            ].copy()

            if sub.empty:
                print(f"[skip] No predictions for {asset} {freq} {dataset}")
                continue

            # Sort by time and ensure expected columns
            sub = sub.sort_values("date")
            # Use model families present in 3.2 for this panel
            models_cls = sorted(sub.loc[sub["task"]=="classification","model"].unique().tolist())
            models_reg = sorted(sub.loc[sub["task"]=="regression","model"].unique().tolist())

            # Realised next-period return (% points) at each timestamp
            y_true_ret_pp = sub[sub["task"]=="regression"][["date","y_true_ret"]].drop_duplicates().set_index("date")["y_true_ret"]
            dates = y_true_ret_pp.index.drop_duplicates() # Remove duplicate dates

            # Rebuild TA signals from Close resampled to this freq; align to dates
            px = resample_close(df_full, asset, freq)
            px = px.reindex(dates, method="ffill").dropna()

            # TA positions
            ta_pos = ta_rules_from_prices(px)
            # Ensure aligned indexes
            for k in ta_pos.keys():
                ta_pos[k] = ta_pos[k].reindex(dates).fillna(0.0)

            # === A) Model-only strategies ===
            # CLS threshold strategies
            for m in models_cls:
                key = (sub["model"]==m) & (sub["task"]=="classification")
                prob = sub.loc[key, ["date","prob_up"]].drop_duplicates().set_index("date")["prob_up"].reindex(dates).astype(float)
                for theta in CLS_THRESHOLDS:
                    pos = cls_threshold_position(prob, theta)
                    # Optional big-move filter using regression predictions if available
                    # Fallback: no filter if pred_ret missing for some models
                    if ("pred_ret" in sub.columns) and (not sub[sub["task"]=="regression"].empty):
                        pred_ret_pp = sub[sub["task"]=="regression"][["date","pred_ret"]].drop_duplicates().set_index("date")["pred_ret"].reindex(dates)
                        pos_bmf = bigmove_filter(pos, pred_ret_pp, BIGMOVE_TAU.get((asset,freq), 0.02))
                    else:
                        pos_bmf = pos.copy()

                    pnl_df = simulate_pnl(dates, pos_bmf, y_true_ret_pp, asset, freq, use_vol_target=USE_VOL_TARGET)
                    row = {
                        "run_id_bt": RUN_ID_BT, "src_run_id": RUN_ID_SRC,
                        "dataset": dataset, "asset": asset, "freq": freq,
                        "strategy_id": f"CLS_{m}_theta{theta}",
                        "family": "ML_CLS",
                        "params": json.dumps({"theta": theta}),
                        "ann_return": ann_return(pnl_df["pnl"], freq),
                        "sharpe": sharpe(pnl_df["pnl"], freq),
                        "max_drawdown": max_drawdown(pnl_df["cum_pnl"]),
                        "turnover": turnover(pnl_df["pos"]).__float__(),
                    }
                    row.update(trade_stats(pnl_df["pos"], pnl_df["pnl"]))
                    row.update(bigmove_stats(y_true_ret_pp, pnl_df["pnl"], asset, freq))
                    results_rows.append(row)

            # REG sized strategies
            for m in models_reg:
                key = (sub["model"]==m) & (sub["task"]=="regression")
                pred = sub.loc[key, ["date","pred_ret"]].drop_duplicates().set_index("date")["pred_ret"].reindex(dates).astype(float)
                for k_val in REG_K_VALUES:
                    pos = reg_sized_position(pred, k_val)
                    pos = bigmove_filter(pos, pred, BIGMOVE_TAU.get((asset,freq), 0.02))
                    pnl_df = simulate_pnl(dates, pos, y_true_ret_pp, asset, freq, use_vol_target=USE_VOL_TARGET)
                    row = {
                        "run_id_bt": RUN_ID_BT, "src_run_id": RUN_ID_SRC,
                        "dataset": dataset, "asset": asset, "freq": freq,
                        "strategy_id": f"REG_{m}_k{k_val}",
                        "family": "ML_REG",
                        "params": json.dumps({"k": k_val}),
                        "ann_return": ann_return(pnl_df["pnl"], freq),
                        "sharpe": sharpe(pnl_df["pnl"], freq),
                        "max_drawdown": max_drawdown(pnl_df["cum_pnl"]),
                        "turnover": turnover(pnl_df["pos"]).__float__(),
                    }
                    row.update(trade_stats(pnl_df["pos"], pnl_df["pnl"]))
                    row.update(bigmove_stats(y_true_ret_pp, pnl_df["pnl"], asset, freq))
                    results_rows.append(row)

            # === B) TA-only strategies (from eng TA) ===
            for ta_name, ta_series in ta_pos.items():
                pnl_df = simulate_pnl(dates, ta_series, y_true_ret_pp, asset, freq, use_vol_target=False)
                row = {
                    "run_id_bt": RUN_ID_BT, "src_run_id": RUN_ID_SRC,
                    "dataset": dataset, "asset": asset, "freq": freq,
                    "strategy_id": ta_name,
                    "family": "TA_ONLY",
                    "params": json.dumps({}),
                    "ann_return": ann_return(pnl_df["pnl"], freq),
                    "sharpe": sharpe(pnl_df["pnl"], freq),
                    "max_drawdown": max_drawdown(pnl_df["cum_pnl"]),
                    "turnover": turnover(pnl_df["pos"]).__float__(),
                }
                row.update(trade_stats(pnl_df["pos"], pnl_df["pnl"]))
                row.update(bigmove_stats(y_true_ret_pp, pnl_df["pnl"], asset, freq))
                results_rows.append(row)

            # === C) Hybrids: Confirm and Weighted ===
            # Use a representative ML CLS and REG stream. Here: best-per-asset later; for now run all.
            for m in models_cls:
                prob = sub[(sub["model"]==m) & (sub["task"]=="classification")][["date","prob_up"]].drop_duplicates().set_index("date")["prob_up"].reindex(dates).astype(float)
                for theta in CLS_THRESHOLDS:
                    pos_ml = cls_threshold_position(prob, theta)
                    for ta_name, pos_ta in ta_pos.items():
                        pos_conf = hybrid_confirm(pos_ml, pos_ta)
                        pnl_df = simulate_pnl(dates, pos_conf, y_true_ret_pp, asset, freq, use_vol_target=USE_VOL_TARGET)
                        row = {
                            "run_id_bt": RUN_ID_BT, "src_run_id": RUN_ID_SRC,
                            "dataset": dataset, "asset": asset, "freq": freq,
                            "strategy_id": f"HYB_CONFIRM_{m}_{ta_name}_theta{theta}",
                            "family": "HYBRID_CONFIRM",
                            "params": json.dumps({"theta": theta}),
                            "ann_return": ann_return(pnl_df["pnl"], freq),
                            "sharpe": sharpe(pnl_df["pnl"], freq),
                            "max_drawdown": max_drawdown(pnl_df["cum_pnl"]),
                            "turnover": turnover(pnl_df["pos"]).__float__(),
                        }
                        row.update(trade_stats(pnl_df["pos"], pnl_df["pnl"]))
                        row.update(bigmove_stats(y_true_ret_pp, pnl_df["pnl"], asset, freq))
                        results_rows.append(row)

            # Weighted hybrids blend REG and TA
            for m in models_reg:
                pred = sub[(sub["model"]==m) & (sub["task"]=="regression")][["date","pred_ret"]].drop_duplicates().set_index("date")["pred_ret"].reindex(dates).astype(float)
                pos_ml = reg_sized_position(pred, REG_K_VALUES[0])  # use baseline k for weight blends
                for ta_name, pos_ta in ta_pos.items():
                    for alpha in HYBRID_WEIGHTS:
                        pos_blend = hybrid_weight(pos_ml, pos_ta, alpha)
                        pnl_df = simulate_pnl(dates, pos_blend, y_true_ret_pp, asset, freq, use_vol_target=USE_VOL_TARGET)
                        row = {
                            "run_id_bt": RUN_ID_BT, "src_run_id": RUN_ID_SRC,
                            "dataset": dataset, "asset": asset, "freq": freq,
                            "strategy_id": f"HYB_WEIGHT_{m}_{ta_name}_a{alpha}",
                            "family": "HYBRID_WEIGHT",
                            "params": json.dumps({"alpha": alpha, "k_base": REG_K_VALUES[0]}),
                            "ann_return": ann_return(pnl_df["pnl"], freq),
                            "sharpe": sharpe(pnl_df["pnl"], freq),
                            "max_drawdown": max_drawdown(pnl_df["cum_pnl"]),
                            "turnover": turnover(pnl_df["pos"]).__float__(),
                        }
                        row.update(trade_stats(pnl_df["pos"], pnl_df["pnl"]))
                        row.update(bigmove_stats(y_true_ret_pp, pnl_df["pnl"], asset, freq))
                        results_rows.append(row)

print("Backtest enumeration complete.")

# === 8) Collate, rank, and write outputs ===

performance_master = pd.DataFrame(results_rows)
if performance_master.empty:
    raise RuntimeError("No backtest results produced.")

# Rank by Sharpe within each asset×freq
def rank_within(df):
    df = df.copy()
    df["rank_sharpe"] = df["sharpe"].rank(ascending=False, method="first")
    return df

performance_master = performance_master.groupby(["dataset","asset","freq"], as_index=False).apply(rank_within).reset_index(drop=True)

# Leaderboard: top N per asset×freq
TOP_N = 5
leaderboard = performance_master.sort_values(["dataset","asset","freq","sharpe"], ascending=[True,True,True,False]).groupby(["dataset","asset","freq"]).head(TOP_N).reset_index(drop=True)

# Save CSVs
performance_master.to_csv(BT_SUM / "performance_master.csv", index=False)
leaderboard.to_csv(BT_SUM / "strategy_leaderboard.csv", index=False)

print("Saved:", BT_SUM / "performance_master.csv")
print("Saved:", BT_SUM / "strategy_leaderboard.csv")

# === 9) Plots: equity curves for top strategies per asset/freq ===
# Rebuild equity for plotting the top picks (keeps runtime reasonable)
def rebuild_equity_for_strategy(strategy_row, df_full_prices, preds_all):
    dataset = strategy_row["dataset"]; asset = strategy_row["asset"]; freq = strategy_row["freq"]
    sid = strategy_row["strategy_id"]; fam = strategy_row["family"]

    sub = preds_all[(preds_all["dataset"]==dataset) & (preds_all["asset"]==asset) & (preds_all["freq"]==freq)].copy().sort_values("date")
    dates = sub[sub["task"]=="regression"][["date","y_true_ret"]].drop_duplicates().set_index("date").index.drop_duplicates() # Remove duplicate dates
    y_true_ret_pp = sub[sub["task"]=="regression"][["date","y_true_ret"]].drop_duplicates().set_index("date")["y_true_ret"]

    # Rebuild TA
    px = resample_close(df_full_prices, asset, freq).reindex(dates, method="ffill").dropna()
    ta = ta_rules_from_prices(px)

    # Recreate positions based on sid
    pos = pd.Series(0.0, index=dates)
    if fam == "ML_CLS":
        m = sid.split("_")[1]
        theta = float(sid.split("theta")[1])
        prob = sub[(sub["model"]==m) & (sub["task"]=="classification")][["date","prob_up"]].drop_duplicates().set_index("date")["prob_up"].reindex(dates)
        pos = cls_threshold_position(prob, theta)
        if "pred_ret" in sub.columns and not sub[sub["task"]=="regression"].empty:
            pred = sub[sub["task"]=="regression"][["date","pred_ret"]].drop_duplicates().set_index("date")["pred_ret"].reindex(dates)
            pos = bigmove_filter(pos, pred, BIGMOVE_TAU.get((asset,freq), 0.02))
    elif fam == "ML_REG":
        m = sid.split("_")[1]
        k_val = float(sid.split("k")[1])
        pred = sub[(sub["model"]==m) & (sub["task"]=="regression")][["date","pred_ret"]].drop_duplicates().set_index("date")["pred_ret"].reindex(dates)
        pos = reg_sized_position(pred, k_val)
        pos = bigmove_filter(pos, pred, BIGMOVE_TAU.get((asset,freq), 0.02))
    elif fam == "TA_ONLY":
        pos = ta.get(sid, pd.Series(0.0, index=dates)).reindex(dates).fillna(0.0)
    elif fam == "HYBRID_CONFIRM":
        _,_, m, ta_name, tail = sid.split("_", 4)
        theta = float(tail.replace("theta",""))
        prob = sub[(sub["model"]==m) & (sub["task"]=="classification")][["date","prob_up"]].drop_duplicates().set_index("date")["prob_up"].reindex(dates)
        pos_ml = cls_threshold_position(prob, theta)
        pos_ta = ta.get(ta_name, pd.Series(0.0, index=dates)).reindex(dates).fillna(0.0)
        pos = hybrid_confirm(pos_ml, pos_ta)
    elif fam == "HYBRID_WEIGHT":
        _,_, m, ta_name, a = sid.split("_", 4)
        alpha = float(a.replace("a",""))
        pred = sub[(sub["model"]==m) & (sub["task"]=="regression")][["date","pred_ret"]].drop_duplicates().set_index("date")["pred_ret"].reindex(dates)
        pos_ml = reg_sized_position(pred, REG_K_VALUES[0])
        pos_ta = ta.get(ta_name, pd.Series(0.0, index=dates)).reindex(dates).fillna(0.0)
        pos = hybrid_weight(pos_ml, pos_ta, alpha)

    pnl_df = simulate_pnl(dates, pos, y_true_ret_pp, asset, freq, use_vol_target=USE_VOL_TARGET)
    return pnl_df

for dataset in DATASETS_TO_USE:
    for asset in ASSETS:
        for freq in FREQS:
            top = leaderboard[(leaderboard["dataset"]==dataset) &
                              (leaderboard["asset"]==asset) &
                              (leaderboard["freq"]==freq)].head(3)
            if top.empty:
                continue
            plt.figure(figsize=(9,5))
            for _, row in top.iterrows():
                pnl_df = rebuild_equity_for_strategy(row, df_full, preds)
                plt.plot(pnl_df.index, pnl_df["cum_pnl"], label=f'{row["strategy_id"]} (Sharpe {row["sharpe"]:.2f})')
            plt.title(f"Equity Curves — {asset} {freq} [{dataset}] (Top 3 by Sharpe)")
            plt.legend()
            plt.tight_layout()
            fname = BT_PLOTS / f"equity_{dataset}_{asset}_{freq}.png"
            plt.savefig(fname)
            plt.close()

print("Top equity plots saved to:", BT_PLOTS)

# === 10) README for 3.3 ===
readme = f"""Run ID: {RUN_ID_BT}
Notebook: 40_backtest_strategies.ipynb
Date: {dt.datetime.now().date().isoformat()}
Mode: Backtest only (reads predictions from run {RUN_ID_SRC})

Overview
--------
This run evaluates trading strategies on the out-of-sample predictions produced by run {RUN_ID_SRC}.
Strategies include:
 - ML (classification threshold; regression-sized; big-move filters)
 - TA-only (RSI(14), SMA 10/50, MACD 12/26/9) computed from ENG close prices
 - Hybrids (ML confirmed by TA; weighted ML–TA blends)

Assumptions
-----------
 - Execution: prediction at t is applied for next period's return (y_true_ret at t)
 - Costs per side (bps): BTC 10, GOLD 3, OIL 3, USDCNY 1
 - Vol targeting: {USE_VOL_TARGET} (target {TARGET_VOL_ANNUAL if USE_VOL_TARGET else 'N/A'})
 - Frequencies: Daily (D) and Weekly (W)
 - Datasets in this run: {', '.join(DATASETS_TO_USE)}

Outputs (60_summary/)
---------------------
 - performance_master.csv       → metrics per strategy × asset × freq
 - strategy_leaderboard.csv     → Top-N strategies by Sharpe per asset×freq

Outputs (70_plots/)
-------------------
 - equity_<dataset>_<asset>_<freq>.png → equity curves for top strategies

Metrics
-------
 - Annualised Return, Sharpe, Max Drawdown
 - Win rate, Trade count, Turnover
 - Big-move skill: precision, recall, PnL share on high-volatility periods

Provenance
----------
 - Source predictions: outputs/runs/{RUN_ID_SRC}/60_summary/predictions_master.csv
 - Inputs (prices for TA): outputs/runs/{RUN_ID_SRC}/00_meta/input/{ENG_FILE.name}

Notes
-----
 - This run does not retrain models. It reads OOS predictions from {RUN_ID_SRC}.
 - Donchian/ATR/ADX strategies are not included due to missing High/Low/ATR columns.
   They can be added if those series are made available.
"""

with open(BT_RUN / "README.txt", "w") as f:
    f.write(readme)

print("README written to:", BT_RUN / "README.txt")
print("✅ Backtest complete.")

Mounted at /content/drive
Reading predictions from: /content/drive/MyDrive/gt-markets/outputs/runs/3.2/60_summary
Writing backtest outputs to: /content/drive/MyDrive/gt-markets/outputs/runs/3.3
Predictions loaded: (180000, 12)


ValueError: cannot reindex on an axis with duplicate labels